# Imports

In [21]:
# Import modules
import tensorflow as tf
import tensorflow_hub as hub

import numpy as np

from keras.applications.mobilenet import MobileNet
from keras.layers import Dropout, Dense, Flatten
from keras.models import Sequential, save_model
from keras.optimizers import Adam
from keras.losses import BinaryCrossentropy

from tqdm import tqdm

In [2]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent.parent))

from preprocessing.audio_processor import preprocess
from config import AUDIO_DIR, AUDIO_CONFIG

🖥️  Using device: cuda


# Data Preprocessing

In [3]:
# Load data parameters
tf.random.set_seed(42)

IMG_HEIGHT = AUDIO_CONFIG["spectrogram_height"]
IMG_WIDTH = AUDIO_CONFIG["spectrogram_width"]
CHANNELS = 3

INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, CHANNELS)

EPOCHS = 5
BATCH_SIZE = 32

In [4]:
audio_data = {
    'training': {
        'spectrograms': [],
        'labels': []
    },
    'testing': {
        'spectrograms': [],
        'labels': []
    }
}

# Track skipped files
skipped_files = []

for split in ['training', 'testing']:
    for label in ['real', 'fake']:
        audio_files = list((AUDIO_DIR / split / label).glob('*.wav')) + list((AUDIO_DIR / split / label).glob('*.mp3'))
        print(f"\nProcessing {label} audio files in {split} set:")
        
        for audio in tqdm(audio_files, desc=f"{split}/{label}"):
            try:
                spectrogram, _ = preprocess(audio)
                
                audio_data[split]['spectrograms'].append(spectrogram)
                audio_data[split]['labels'].append(0 if label=='real' else 1)
            except Exception as e:
                # Skip corrupted or unreadable files
                skipped_files.append((str(audio), str(e)))
                continue

# Report skipped files
if skipped_files:
    print(f"\nSkipped {len(skipped_files)} corrupted/unreadable files:")
    for file_path, error in skipped_files:
        print(f"  - {file_path}")
else:
    print(f"\nAll files processed successfully!")



Processing real audio files in training set:


training/real: 100%|██████████| 4000/4000 [01:06<00:00, 60.49it/s] 



Processing fake audio files in training set:


training/fake:  66%|██████▌   | 2647/4000 [00:24<00:12, 108.34it/s]d:\OneDrive\Documents\Julien\Documents\!ESILV\A5\!S9\Explainability AI\Project\XAI_Final_Project\preprocessing\audio_processor.py:30: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, sr=None)
d:\OneDrive\Documents\Julien\Documents\!ESILV\A5\.env_A5\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
training/fake: 100%|██████████| 4000/4000 [00:37<00:00, 106.15it/s]



Processing real audio files in testing set:


testing/real: 100%|██████████| 1000/1000 [00:14<00:00, 67.47it/s]



Processing fake audio files in testing set:


testing/fake: 100%|██████████| 1000/1000 [00:10<00:00, 97.87it/s]


Skipped 1 corrupted/unreadable files:
  - d:\OneDrive\Documents\Julien\Documents\!ESILV\A5\!S9\Explainability AI\Project\XAI_Final_Project\data\audio\training\fake\file13424.mp3


In [5]:
# Convert lists to NumPy arrays and create TensorFlow datasets
train_tensors = np.array(audio_data['training']['spectrograms'])
train_labels = np.array(audio_data['training']['labels'])

test_tensors = np.array(audio_data['testing']['spectrograms'])
test_labels = np.array(audio_data['testing']['labels'])

print(f"Train shape: {train_tensors.shape}, Labels: {train_labels.shape}")
print(f"Test shape: {test_tensors.shape}, Labels: {test_labels.shape}")

train_ds = tf.data.Dataset.from_tensor_slices((train_tensors, train_labels)).shuffle(1000).batch(BATCH_SIZE)
test_ds = tf.data.Dataset.from_tensor_slices((test_tensors, test_labels)).shuffle(100).batch(BATCH_SIZE)

print(f"\nTrain batches: {len(train_ds)}")
print(f"Test batches: {len(test_ds)}")

Train shape: (7999, 224, 224, 3), Labels: (7999,)
Test shape: (2000, 224, 224, 3), Labels: (2000,)

Train batches: 250
Test batches: 63


In [6]:
import gc

del audio_data, train_tensors, train_labels, test_tensors, test_labels

gc.collect()

1240

# Model

In [19]:
mobilenet = MobileNet(input_shape=INPUT_SHAPE, include_top=False, weights='imagenet')

# Do not train the model, leverage pretrained layers
for layer in mobilenet.layers:
    layer.trainable = False

In [22]:
model_mobilenet = Sequential()

model_mobilenet.add(mobilenet)
model_mobilenet.add(Flatten())
model_mobilenet.add(Dense(1024, activation='relu'))
model_mobilenet.add(Dropout(0.3))
model_mobilenet.add(Dense(1024, activation='relu'))
model_mobilenet.add(Dropout(0.3))
model_mobilenet.add(Dense(1, activation='sigmoid'))

# Compile the model and define folder to store logs
model_mobilenet.compile(optimizer=Adam(), 
                     loss=BinaryCrossentropy(from_logits=False), 
                     metrics=['accuracy'])

In [23]:
history = model_mobilenet.fit(
    train_ds,
    validation_data = test_ds,
    shuffle=True,
    epochs = EPOCHS
)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 355s 1s/step - accuracy: 0.9431 - loss: 3.1833 - val_accuracy: 0.5755 - val_loss: 1.4315
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 185s 739ms/step - accuracy: 0.9570 - loss: 0.1756 - val_accuracy: 0.5710 - val_loss: 2.4713
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 186s 744ms/step - accuracy: 0.9572 - loss: 0.2333 - val_accuracy: 0.8390 - val_loss: 0.4844
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 186s 743ms/step - accuracy: 0.9624 - loss: 0.1321 - val_accuracy: 0.8600 - val_loss: 0.4082
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 186s 744ms/step - accuracy: 0.9612 - loss: 0.1372 - val_accuracy: 0.8865 - val_loss: 0.2873


In [24]:
save_model(model_mobilenet, filepath="./mobilenet.h5")